In [0]:
df = spark.read.format('parquet')\
    .load('abfss://bronze@datalakeete1.dfs.core.windows.net/orders')

df.display()

In [0]:
df.printSchema()

In [0]:
df = df.withColumnRenamed('_rescued_data','rescued_data')

In [0]:
df.display()

In [0]:
df= df.drop('rescued_data')

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import to_timestamp, col,year

In [0]:
df = df.withColumn('order_date',to_timestamp(col('order_date')))
df.display()

In [0]:
df= df.withColumn('year',year(col("order_date")))

In [0]:
%python
df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, desc
df = df.withColumn('flag',dense_rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))

In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, desc,rank
df = df.withColumn('rank_flag',rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))

In [0]:
%python
df.display()


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, desc,rank,row_number
df = df.withColumn('row_flag',row_number().over(Window.partitionBy('year').orderBy(desc('total_amount'))))

In [0]:
df.display()

### ***Classes - OOP***

In [0]:
class Windows:
    def dense_rank(self,df):
        df_dense_rank = df.withColumn('flag',dense_rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df_dense_rank
    
    def rank(self,df):
        df_rank=df.withColumn('rank_flag',rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df_rank
    
    def row_number(self,df):
        df_row_number=df.withColumn('row_flag',row_number().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df_row_number

In [0]:
%python
df_new = df

In [0]:
df_new

In [0]:
df_new.display()

In [0]:
obj = Windows()

In [0]:
df_result=obj.dense_rank(df_new)

In [0]:
df_result.display()

### ***Data Writing***

In [0]:
df.write.format('delta').mode('append').save('abfss://silver@datalakeete1.dfs.core.windows.net/orders')